In [ ]:
!pip install -q gradio

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

print("Searching for saved models...\n")

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file in [
            "crop_recommendation_model.pkl",
            "crop_yield_prediction_model.pkl",
            "weather_risk_xgboost_model.pkl",
            "final_rl_irrigation_ppo_model.zip"
        ]:
            print(os.path.join(root, file))

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install -q stable-baselines3 gymnasium
import joblib
from stable_baselines3 import PPO

crop_model_path = "/content/drive/MyDrive/crop_recommendation_model.pkl"
yield_model_path = "/content/drive/MyDrive/crop_yield_prediction_model.pkl"
weather_model_path = "/content/drive/MyDrive/weather_risk_xgboost_model.pkl"
rl_model_path = "/content/drive/MyDrive/final_rl_irrigation_ppo_model.zip"

# Load all models
crop_model = joblib.load(crop_model_path)
yield_model = joblib.load(yield_model_path)
weather_model = joblib.load(weather_model_path)
rl_model = PPO.load(rl_model_path)

print("✅ All 4 models loaded successfully!")

print("Crop model:", type(crop_model))
print("Yield model:", type(yield_model))
print("Weather model:", type(weather_model))
print("RL model:", type(rl_model))

In [ ]:
import numpy as np
import pandas as pd

weather_risk_mapping = {
    0: "Extreme",
    1: "High",
    2: "Low",
    3: "Moderate"
}

def agriculture_ai_prediction(
    N, P, K,
    temperature, humidity, ph, rainfall,
    soil_moisture, soil_temperature,
    crop_year, area, fertilizer, pesticide,
    season, state,
    month, wind_speed, air_pressure
):

    # -----------------------------
    # 1. Crop Recommendation
    # -----------------------------
    crop_input = pd.DataFrame([{
        "N": N,
        "P": P,
        "K": K,
        "temperature": temperature,
        "humidity": humidity,
        "ph": ph,
        "rainfall": rainfall
    }])

    recommended_crop = crop_model.predict(crop_input)[0]

    # -----------------------------
    # 2. Crop Yield Prediction
    # -----------------------------
    yield_input = pd.DataFrame([{
        "Crop": recommended_crop,
        "Season": season,
        "State": state,
        "Crop_Year": crop_year,
        "Area": area,
        "Annual_Rainfall": rainfall,
        "Fertilizer": fertilizer,
        "Pesticide": pesticide
    }])

    predicted_yield = float(yield_model.predict(yield_input)[0])

    # -----------------------------
    # 3. Weather Risk Prediction
    # -----------------------------
    month_number = {
        "January": 1,
        "February": 2,
        "March": 3,
        "April": 4,
        "May": 5,
        "June": 6,
        "July": 7,
        "August": 8,
        "September": 9,
        "October": 10,
        "November": 11,
        "December": 12
    }

    m = month_number[month]

    weather_input = pd.DataFrame([{
        "month": month,
        "season": season,
        "state": state,
        "avg_temp": temperature,
        "min_temp": temperature - 3,
        "max_temp": temperature + 3,
        "wind_speed": wind_speed,
        "air_pressure": air_pressure,
        "elevation": 500,
        "latitude": 17.38,
        "longitude": 78.48,
        "previous_rainfall": rainfall,
        "previous_avg_temp": temperature,
        "previous_wind_speed": wind_speed,
        "previous_pressure": air_pressure,
        "rainfall_3day": rainfall,
        "rainfall_7day": rainfall,
        "rainfall_3day_avg": rainfall / 3,
        "rainfall_7day_avg": rainfall / 7,
        "temp_3day_avg": temperature,
        "temp_7day_avg": temperature,
        "pressure_change": 0,
        "temp_change": 0,
        "wind_change": 0,
        "day_of_year": int((m - 1) * 30 + 15),
        "month_sin": np.sin(2 * np.pi * m / 12),
        "month_cos": np.cos(2 * np.pi * m / 12)
    }])

    weather_code = int(weather_model.predict(weather_input)[0])
    weather_risk = weather_risk_mapping.get(
        weather_code,
        str(weather_code)
    )

    # -----------------------------
    # 4. RL Irrigation
    # -----------------------------
    # Normalize inputs to approximately 0-1
    rl_state = np.array([[
        np.clip(soil_moisture / 1000, 0, 1),
        np.clip(soil_temperature / 50, 0, 1),
        np.clip(temperature / 50, 0, 1),
        np.clip(humidity / 100, 0, 1),
        np.clip(rainfall / 1000, 0, 1),
        np.clip(wind_speed / 50, 0, 1)
    ]], dtype=np.float32)

    action, _ = rl_model.predict(
        rl_state,
        deterministic=True
    )

    irrigation = (
        "IRRIGATION"
        if int(action) == 1
        else "NO IRRIGATION"
    )

    return {
        "Recommended Crop": recommended_crop,
        "Predicted Yield": round(predicted_yield, 2),
        "Weather Risk": weather_risk,
        "Irrigation": irrigation
    }

print("✅ Integrated prediction function created.")

In [ ]:
# Test 1 - Cool, dry conditions
result_1 = agriculture_ai_prediction(
    N=20,
    P=10,
    K=10,
    temperature=15.0,
    humidity=30.0,
    ph=4.5,
    rainfall=10.0,
    soil_moisture=100.0,
    soil_temperature=15.0,
    crop_year=2020,
    area=10.0,
    fertilizer=20.0,
    pesticide=2.0,
    season="Winter",
    state="Punjab",
    month="January",
    wind_speed=5.0,
    air_pressure=1000.0
)

# Test 2 - Hot, wet conditions
result_2 = agriculture_ai_prediction(
    N=150,
    P=100,
    K=150,
    temperature=45.0,
    humidity=95.0,
    ph=8.0,
    rainfall=500.0,
    soil_moisture=800.0,
    soil_temperature=45.0,
    crop_year=2026,
    area=1000.0,
    fertilizer=1000.0,
    pesticide=200.0,
    season="Monsoon",
    state="Kerala",
    month="July",
    wind_speed=50.0,
    air_pressure=800.0
)

print("========== TEST 1 ==========")
print(result_1)

print("\n========== TEST 2 ==========")
print(result_2)

In [ ]:
import gradio as gr

def gradio_prediction(
    N, P, K,
    temperature, humidity, ph, rainfall,
    soil_moisture, soil_temperature,
    crop_year, area, fertilizer, pesticide,
    season, state, month,
    wind_speed, air_pressure
):

    result = agriculture_ai_prediction(
        N, P, K,
        temperature, humidity, ph, rainfall,
        soil_moisture, soil_temperature,
        crop_year, area, fertilizer, pesticide,
        season, state, month,
        wind_speed, air_pressure
    )

    return (
        result["Recommended Crop"],
        result["Predicted Yield"],
        result["Weather Risk"],
        result["Irrigation"]
    )


with gr.Blocks(title="Agriculture AI Decision Support System") as demo:

    gr.Markdown(
        """
        # 🌾 Agriculture AI Decision Support System

        AI-powered crop recommendation, yield prediction,
        weather risk assessment and irrigation optimization.
        """
    )

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🌱 Farm & Environmental Inputs")

            N = gr.Number(label="Nitrogen (N)", value=78)
            P = gr.Number(label="Phosphorus (P)", value=45)
            K = gr.Number(label="Potassium (K)", value=48)

            temperature = gr.Number(label="Temperature (°C)", value=38)
            humidity = gr.Number(label="Humidity (%)", value=80)
            ph = gr.Number(label="Soil pH", value=5.58)
            rainfall = gr.Number(label="Rainfall (mm)", value=150)

            soil_moisture = gr.Number(label="Soil Moisture", value=250)
            soil_temperature = gr.Number(
                label="Soil Temperature (°C)",
                value=30
            )

        with gr.Column():
            gr.Markdown("### 🌾 Crop & Location Information")

            crop_year = gr.Number(label="Crop Year", value=2026)
            area = gr.Number(label="Area", value=100)
            fertilizer = gr.Number(label="Fertilizer", value=100)
            pesticide = gr.Number(label="Pesticide", value=10)

            season = gr.Dropdown(
                ["Kharif", "Rabi", "Whole Year", "Summer", "Winter"],
                label="Season",
                value="Kharif"
            )

            state = gr.Dropdown(
                [
                    "Telangana",
                    "Andhra Pradesh",
                    "Punjab",
                    "Kerala",
                    "Tamil Nadu",
                    "Karnataka",
                    "Maharashtra"
                ],
                label="State",
                value="Telangana"
            )

            month = gr.Dropdown(
                [
                    "January", "February", "March", "April",
                    "May", "June", "July", "August",
                    "September", "October", "November", "December"
                ],
                label="Month",
                value="September"
            )

            wind_speed = gr.Number(
                label="Wind Speed",
                value=15
            )

            air_pressure = gr.Number(
                label="Air Pressure",
                value=1200
            )

    generate_btn = gr.Button(
        "🚀 Generate Agriculture AI Recommendation",
        variant="primary"
    )

    gr.Markdown("## 📊 Agriculture AI Results")

    with gr.Row():
        crop_output = gr.Textbox(
            label="🌱 Recommended Crop"
        )

        yield_output = gr.Number(
            label="📈 Predicted Yield"
        )

        weather_output = gr.Textbox(
            label="🌦️ Weather Risk"
        )

        irrigation_output = gr.Textbox(
            label="💧 Irrigation Decision"
        )

    generate_btn.click(
        fn=gradio_prediction,
        inputs=[
            N, P, K,
            temperature, humidity, ph, rainfall,
            soil_moisture, soil_temperature,
            crop_year, area, fertilizer, pesticide,
            season, state, month,
            wind_speed, air_pressure
        ],
        outputs=[
            crop_output,
            yield_output,
            weather_output,
            irrigation_output
        ]
    )

print("✅ Gradio interface created successfully.")

In [ ]:
demo.launch(
    share=True,
    debug=True
)

In [ ]:
import os

print("Searching for feature-name files...\n")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if "feature" in file.lower() or "yield" in file.lower():
            print(os.path.join(root, file))

In [ ]:
# Create missing feature-name files for integration

import joblib

MODEL_DIR = "/content/drive/MyDrive"

# Crop Recommendation features
crop_features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

# Crop Yield Prediction features
yield_features = [
    "Crop",
    "Season",
    "State",
    "Crop_Year",
    "Area",
    "Annual_Rainfall",
    "Fertilizer",
    "Pesticide"
]

# Save them to Google Drive
joblib.dump(
    crop_features,
    f"{MODEL_DIR}/crop_feature_names.pkl"
)

joblib.dump(
    yield_features,
    f"{MODEL_DIR}/yield_feature_names.pkl"
)

print("Feature-name files created successfully!")
print("\nCreated:")
print("✓ crop_feature_names.pkl")
print("✓ yield_feature_names.pkl")

In [ ]:
!pip install -q stable-baselines3 gymnasium

In [ ]:
# Recreate Weather Risk feature names

import joblib

MODEL_DIR = "/content/drive/MyDrive"

weather_features = [
    "month",
    "season",
    "state",
    "avg_temp",
    "min_temp",
    "max_temp",
    "wind_speed",
    "air_pressure",
    "elevation",
    "latitude",
    "longitude",
    "previous_rainfall",
    "previous_avg_temp",
    "previous_wind_speed",
    "previous_pressure",
    "rainfall_3day",
    "rainfall_7day",
    "rainfall_3day_avg",
    "rainfall_7day_avg",
    "temp_3day_avg",
    "temp_7day_avg",
    "pressure_change",
    "temp_change",
    "wind_change",
    "day_of_year",
    "month_sin",
    "month_cos"
]

joblib.dump(
    weather_features,
    f"{MODEL_DIR}/weather_risk_feature_names.pkl"
)

print("Weather feature-name file created successfully!")
print(f"Number of features: {len(weather_features)}")
print("Saved as: weather_risk_feature_names.pkl")

In [ ]:
# Recreate Weather Risk label encoder

import joblib
from sklearn.preprocessing import LabelEncoder

MODEL_DIR = "/content/drive/MyDrive"

# Original weather risk classes
weather_classes = [
    "Extreme",
    "High",
    "Low",
    "Moderate"
]

# Create encoder with the original class ordering
weather_encoder = LabelEncoder()
weather_encoder.fit(weather_classes)

# Save encoder
joblib.dump(
    weather_encoder,
    f"{MODEL_DIR}/weather_risk_label_encoder.pkl"
)

print("Weather label encoder created successfully!")
print("Classes:", list(weather_encoder.classes_))
print("Saved as: weather_risk_label_encoder.pkl")

In [ ]:
# ============================================================
# Load all 7 Agriculture AI modules
# ============================================================

import joblib
from stable_baselines3 import PPO

MODEL_DIR = "/content/drive/MyDrive"

# 1. Crop Recommendation
crop_model = joblib.load(
    f"{MODEL_DIR}/crop_recommendation_model.pkl"
)
crop_features = joblib.load(
    f"{MODEL_DIR}/crop_feature_names.pkl"
)

# 2. Crop Yield Prediction
yield_model = joblib.load(
    f"{MODEL_DIR}/crop_yield_prediction_model.pkl"
)
yield_features = joblib.load(
    f"{MODEL_DIR}/yield_feature_names.pkl"
)

# 3. Weather Risk
weather_model = joblib.load(
    f"{MODEL_DIR}/weather_risk_xgboost_model.pkl"
)
weather_features = joblib.load(
    f"{MODEL_DIR}/weather_risk_feature_names.pkl"
)
weather_encoder = joblib.load(
    f"{MODEL_DIR}/weather_risk_label_encoder.pkl"
)

# 4. RL Irrigation
rl_model = PPO.load(
    f"{MODEL_DIR}/final_rl_irrigation_ppo_model.zip"
)

# 5. Market Price Forecasting
market_xgb_model = joblib.load(
    f"{MODEL_DIR}/market_price_xgboost_model.pkl"
)
market_trend_model = joblib.load(
    f"{MODEL_DIR}/market_price_trend_model.pkl"
)
market_features = joblib.load(
    f"{MODEL_DIR}/market_price_feature_names.pkl"
)
market_info = joblib.load(
    f"{MODEL_DIR}/market_price_model_info.pkl"
)

# 6. Climate Change
climate_model = joblib.load(
    f"{MODEL_DIR}/climate_change_yield_model.pkl"
)
climate_features = joblib.load(
    f"{MODEL_DIR}/climate_change_feature_names.pkl"
)
climate_info = joblib.load(
    f"{MODEL_DIR}/climate_change_model_info.pkl"
)

# 7. Sustainability
sustainability_config = joblib.load(
    f"{MODEL_DIR}/sustainability_score_config.pkl"
)
sustainability_info = joblib.load(
    f"{MODEL_DIR}/sustainability_model_info.pkl"
)

print("==========================================")
print("AGRICULTURE AI — 7 MODULES")
print("==========================================")
print("✓ Crop Recommendation")
print("✓ Crop Yield Prediction")
print("✓ Weather Risk Prediction")
print("✓ RL Irrigation Optimization")
print("✓ Market Price Forecasting")
print("✓ Climate Change Scenario Analysis")
print("✓ Sustainability Score")
print("------------------------------------------")
print("All 7 modules loaded successfully!")

In [ ]:
# ============================================================
# Unified Agriculture AI Prediction Function
# ============================================================

def agriculture_ai_7_module_prediction(
    N, P, K,
    temperature,
    humidity,
    ph,
    rainfall,
    soil_moisture,
    soil_temperature,
    crop_year,
    area,
    fertilizer,
    pesticide,
    season,
    state,
    month,
    wind_speed,
    air_pressure
):

    # --------------------------------------------------------
    # 1. CROP RECOMMENDATION
    # --------------------------------------------------------

    crop_input = pd.DataFrame([[
        N, P, K,
        temperature,
        humidity,
        ph,
        rainfall
    ]], columns=crop_features)

    recommended_crop = crop_model.predict(crop_input)[0]

    # --------------------------------------------------------
    # 2. CROP YIELD PREDICTION
    # --------------------------------------------------------

    yield_input = pd.DataFrame([{
        "Crop": recommended_crop,
        "Season": season,
        "State": state,
        "Crop_Year": crop_year,
        "Area": area,
        "Annual_Rainfall": rainfall,
        "Fertilizer": fertilizer,
        "Pesticide": pesticide
    }])

    yield_input = yield_input[yield_features]

    predicted_yield = float(
        yield_model.predict(yield_input)[0]
    )

    # --------------------------------------------------------
    # 3. WEATHER RISK
    # --------------------------------------------------------

    month_map = {
        "January": 1,
        "February": 2,
        "March": 3,
        "April": 4,
        "May": 5,
        "June": 6,
        "July": 7,
        "August": 8,
        "September": 9,
        "October": 10,
        "November": 11,
        "December": 12
    }

    month_num = month_map.get(month, 1)

    day_of_year = int(
        (month_num - 1) * 30.4 + 15
    )

    weather_input = pd.DataFrame([{
        "month": month_num,
        "season": season,
        "state": state,

        "avg_temp": temperature,
        "min_temp": temperature - 3,
        "max_temp": temperature + 3,

        "wind_speed": wind_speed,
        "air_pressure": air_pressure,

        "elevation": 500,
        "latitude": 17.38,
        "longitude": 78.48,

        "previous_rainfall": rainfall,
        "previous_avg_temp": temperature,
        "previous_wind_speed": wind_speed,
        "previous_pressure": air_pressure,

        "rainfall_3day": rainfall,
        "rainfall_7day": rainfall,

        "rainfall_3day_avg": rainfall,
        "rainfall_7day_avg": rainfall,

        "temp_3day_avg": temperature,
        "temp_7day_avg": temperature,

        "pressure_change": 0,
        "temp_change": 0,
        "wind_change": 0,

        "day_of_year": day_of_year,

        "month_sin": np.sin(
            2 * np.pi * month_num / 12
        ),

        "month_cos": np.cos(
            2 * np.pi * month_num / 12
        )
    }])

    weather_input = weather_input[weather_features]

    weather_code = int(
        weather_model.predict(weather_input)[0]
    )

    weather_risk = weather_encoder.inverse_transform(
        [weather_code]
    )[0]

    # --------------------------------------------------------
    # 4. RL IRRIGATION
    # --------------------------------------------------------

    irrigation_state = np.array([[
        soil_moisture / 1000,
        soil_temperature / 50,
        temperature / 50,
        humidity / 100,
        rainfall / 1000,
        wind_speed / 50
    ]], dtype=np.float32)

    irrigation_state = np.clip(
        irrigation_state,
        0,
        1
    )

    irrigation_action, _ = rl_model.predict(
        irrigation_state,
        deterministic=True
    )

    irrigation_decision = (
        "IRRIGATION"
        if int(irrigation_action) == 1
        else "NO IRRIGATION"
    )

    # --------------------------------------------------------
    # 5. MARKET PRICE FORECASTING
    # --------------------------------------------------------
    #
    # The trained market model forecasts Wheat price for
    # Varanasi(Grain). Its historical feature vector is
    # time-series based.
    #
    # We use the latest known price from the model's training
    # information when the complete historical series is not
    # supplied through the UI.
    # --------------------------------------------------------

    market_target = market_info.get(
        "target",
        "Modal_Price"
    )

    market_unit = market_info.get(
        "unit",
        "Rs./Quintal"
    )

    market_forecast = "Available from Market Price Model"

    # --------------------------------------------------------
    # 6. CLIMATE CHANGE SCENARIO
    # --------------------------------------------------------

    climate_base = pd.DataFrame([{
        "Crop": recommended_crop,
        "Crop_Year": crop_year,
        "Season": season,
        "State": state,
        "Annual_Rainfall": rainfall
    }])

    climate_base = climate_base[climate_features]

    baseline_climate_yield = float(
        climate_model.predict(climate_base)[0]
    )

    moderate_climate = climate_base.copy()
    moderate_climate["Annual_Rainfall"] = rainfall * 0.90

    severe_climate = climate_base.copy()
    severe_climate["Annual_Rainfall"] = rainfall * 0.75

    moderate_yield = float(
        climate_model.predict(moderate_climate)[0]
    )

    severe_yield = float(
        climate_model.predict(severe_climate)[0]
    )

    moderate_change = (
        (moderate_yield - baseline_climate_yield)
        / baseline_climate_yield
    ) * 100

    severe_change = (
        (severe_yield - baseline_climate_yield)
        / baseline_climate_yield
    ) * 100

    # --------------------------------------------------------
    # 7. SUSTAINABILITY SCORE
    # --------------------------------------------------------

    fertilizer_per_ha = (
        fertilizer / area
        if area > 0 else 0
    )

    pesticide_per_ha = (
        pesticide / area
        if area > 0 else 0
    )

    production_per_ha = predicted_yield

    cfg = sustainability_config

    fert_range = (
        cfg["fertilizer_max"]
        - cfg["fertilizer_min"]
    )

    pest_range = (
        cfg["pesticide_max"]
        - cfg["pesticide_min"]
    )

    production_range = (
        cfg["production_max"]
        - cfg["production_min"]
    )

    fert_score = 100 * (
        1 -
        (
            fertilizer_per_ha -
            cfg["fertilizer_min"]
        ) / fert_range
    )

    pest_score = 100 * (
        1 -
        (
            pesticide_per_ha -
            cfg["pesticide_min"]
        ) / pest_range
    )

    efficiency_score = 100 * (
        (
            production_per_ha -
            cfg["production_min"]
        ) / production_range
    )

    sustainability_score = (
        0.35 * fert_score +
        0.30 * pest_score +
        0.35 * efficiency_score
    )

    sustainability_score = float(
        np.clip(
            sustainability_score,
            0,
            100
        )
    )

    if sustainability_score >= 60:
        sustainability_category = "High Sustainability"
    elif sustainability_score >= 30:
        sustainability_category = "Moderate Sustainability"
    else:
        sustainability_category = "Low Sustainability"

    # --------------------------------------------------------
    # FINAL RESULTS
    # --------------------------------------------------------

    return {
        "Recommended Crop": recommended_crop,

        "Predicted Yield": round(
            predicted_yield,
            4
        ),

        "Weather Risk": weather_risk,

        "Irrigation Decision": irrigation_decision,

        "Market Price Forecast":
            market_forecast,

        "Market Unit": market_unit,

        "Climate Baseline Yield":
            round(baseline_climate_yield, 4),

        "Climate Moderate Scenario":
            round(moderate_yield, 4),

        "Climate Moderate Change %":
            round(moderate_change, 4),

        "Climate Severe Scenario":
            round(severe_yield, 4),

        "Climate Severe Change %":
            round(severe_change, 4),

        "Sustainability Score":
            round(sustainability_score, 2),

        "Sustainability Category":
            sustainability_category
    }


print("Unified 7-module prediction function created successfully!")

In [ ]:
import traceback

try:
    test_result = agriculture_ai_7_module_prediction(
        N=90,
        P=42,
        K=43,
        temperature=25.0,
        humidity=80.0,
        ph=6.5,
        rainfall=200.0,
        soil_moisture=200.0,
        soil_temperature=25.0,
        crop_year=2024,
        area=1000.0,
        fertilizer=50000.0,
        pesticide=500.0,
        season="Kharif",
        state="Andhra Pradesh",
        month=7,
        wind_speed=10.0,
        air_pressure=1010.0
    )

    print("TEST SUCCESSFUL")
    print(test_result)

except Exception as e:
    print("ERROR FOUND")
    print("Error type:", type(e).__name__)
    print("Error:", e)
    print("\nFULL TRACEBACK:")
    traceback.print_exc()

In [ ]:
# ============================================================
# INSPECT WEATHER MODEL CATEGORICAL VALUES
# ============================================================

weather_model = joblib.load(
    "/content/drive/MyDrive/weather_risk_xgboost_model.pkl"
)

print("Weather model type:", type(weather_model))

print("\nWeather model steps:")
print(weather_model.named_steps.keys())

preprocessor = weather_model.named_steps.get("preprocessor")

print("\nPreprocessor:")
print(preprocessor)

print("\nCategorical encoder information:")

for name, transformer, columns in preprocessor.transformers_:
    if name == "cat":
        print("Categorical columns:")
        print(columns)

        encoder = transformer.named_steps.get("onehot")

        if encoder is not None:
            print("\nEncoder categories:")
            for col, categories in zip(columns, encoder.categories_):
                print(f"\n{col}:")
                print(categories)

In [ ]:
# Show exactly what the Weather model expects

preprocessor = weather_model.named_steps["preprocessor"]

for name, transformer, columns in preprocessor.transformers_:
    if name == "categorical":
        encoder = transformer

        print("Categorical columns:", columns)
        print("\nExpected categories:")

        for col, categories in zip(columns, encoder.categories_):
            print(f"\n{col}")
            print("Type:", categories.dtype)
            print("Values:", categories)

In [ ]:
# ============================================================
# CHECK MARKET PRICE MODELS
# ============================================================

import joblib
import os

MODEL_DIR = "/content/drive/MyDrive"

market_xgb_path = f"{MODEL_DIR}/market_price_xgboost_model.pkl"
market_trend_path = f"{MODEL_DIR}/market_price_trend_model.pkl"
market_features_path = f"{MODEL_DIR}/market_price_feature_names.pkl"

print("Market model files:")

for path in [
    market_xgb_path,
    market_trend_path,
    market_features_path
]:
    print(
        "FOUND" if os.path.exists(path) else "MISSING",
        "->",
        path
    )

if os.path.exists(market_xgb_path):
    market_price_xgb_model = joblib.load(market_xgb_path)
    print("\nXGBoost model loaded:")
    print(type(market_price_xgb_model))

if os.path.exists(market_trend_path):
    market_price_trend_model = joblib.load(market_trend_path)
    print("\nTrend model loaded:")
    print(type(market_price_trend_model))

if os.path.exists(market_features_path):
    market_price_features = joblib.load(market_features_path)
    print("\nFeature names:")
    print(market_price_features)

In [ ]:
# Check Market Price model directly

print("Market Price Model Check")
print("-------------------------")

print("Model:", type(market_price_xgb_model))
print("Features:", market_price_features)

month_number = 1
day_of_year = 15

market_input = np.array([[
    0.0,   # res_lag_1
    0.0,   # res_lag_7
    0.0,   # res_lag_14
    0.0,   # res_lag_30
    0.0,   # res_rolling_7
    0.0,   # res_rolling_14
    month_number,
    day_of_year
]])

market_prediction = market_price_xgb_model.predict(market_input)[0]

print("Market Price Forecast:", float(market_prediction))
print("Market Price model working successfully!")

In [ ]:

# ============================================================
# FIX MARKET PRICE FORECAST IN 7-MODULE FUNCTION
# ============================================================

def get_market_price_forecast(month_number=1, day_of_year=15):

    market_input = np.array([[
        0.0,  # res_lag_1
        0.0,  # res_lag_7
        0.0,  # res_lag_14
        0.0,  # res_lag_30
        0.0,  # res_rolling_7
        0.0,  # res_rolling_14
        float(month_number),
        float(day_of_year)
    ]])

    prediction = market_price_xgb_model.predict(market_input)[0]

    return float(prediction)


# Test
market_test = get_market_price_forecast(
    month_number=1,
    day_of_year=15
)

print("Market Price Forecast:", market_test)
print("Market Price integration function working successfully!")

In [ ]:
# ============================================================
# CORRECTED UNIFIED 7-MODULE AGRICULTURE AI FUNCTION
# ============================================================

def agriculture_ai_7_module_prediction(
    N, P, K,
    temperature, humidity, ph, rainfall,
    soil_moisture, soil_temperature,
    crop_year, area, fertilizer, pesticide,
    season, state, month,
    wind_speed, air_pressure
):

    # --------------------------------------------------------
    # 1. CROP RECOMMENDATION
    # --------------------------------------------------------

    crop_input = pd.DataFrame([{
        "N": N,
        "P": P,
        "K": K,
        "temperature": temperature,
        "humidity": humidity,
        "ph": ph,
        "rainfall": rainfall
    }])

    recommended_crop = crop_model.predict(crop_input)[0]


    # --------------------------------------------------------
    # 2. CROP YIELD PREDICTION
    # --------------------------------------------------------

    yield_input = pd.DataFrame([{
        "Crop": recommended_crop,
        "Crop_Year": crop_year,
        "Season": season,
        "State": state,
        "Area": area,
        "Annual_Rainfall": rainfall,
        "Fertilizer": fertilizer,
        "Pesticide": pesticide
    }])

    predicted_yield = yield_model.predict(yield_input)[0]


    # --------------------------------------------------------
    # 3. WEATHER RISK PREDICTION
    # --------------------------------------------------------

    state_codes = {
        "Andhra Pradesh": "AP",
        "Arunachal Pradesh": "AR",
        "Assam": "AS",
        "Bihar": "BR",
        "Chhattisgarh": "CT",
        "Goa": "GA",
        "Gujarat": "GJ",
        "Haryana": "HR",
        "Himachal Pradesh": "HP",
        "Jharkhand": "JH",
        "Karnataka": "KA",
        "Kerala": "KL",
        "Madhya Pradesh": "MP",
        "Maharashtra": "MH",
        "Manipur": "MN",
        "Meghalaya": "ML",
        "Mizoram": "MZ",
        "Nagaland": "NL",
        "Odisha": "OR",
        "Punjab": "PB",
        "Rajasthan": "RJ",
        "Sikkim": "SK",
        "Tamil Nadu": "TN",
        "Telangana": "TG",
        "Tripura": "TR",
        "Uttar Pradesh": "UP",
        "Uttarakhand": "UK",
        "West Bengal": "WB",
        "Delhi": "DL",
        "Jammu and Kashmir": "JK",
        "Puducherry": "PY",
        "Andaman and Nicobar Islands": "AN"
    }

    weather_state = state_codes.get(state, state)

    month_names = {
        1: "January",
        2: "February",
        3: "March",
        4: "April",
        5: "May",
        6: "June",
        7: "July",
        8: "August",
        9: "September",
        10: "October",
        11: "November",
        12: "December"
    }

    if isinstance(month, (int, np.integer)):
        weather_month = month_names.get(int(month), "July")
    else:
        weather_month = str(month)

    season_map = {
        "Kharif": "Monsoon",
        "Rabi": "Winter",
        "Whole Year": "Summer",
        "Autumn": "Post-monsoon",
        "Summer": "Summer",
        "Winter": "Winter",
        "Monsoon": "Monsoon",
        "Post-monsoon": "Post-monsoon"
    }

    weather_season = season_map.get(season, season)

    month_number_map = {
        "January": 1,
        "February": 2,
        "March": 3,
        "April": 4,
        "May": 5,
        "June": 6,
        "July": 7,
        "August": 8,
        "September": 9,
        "October": 10,
        "November": 11,
        "December": 12
    }

    month_number = month_number_map.get(weather_month, 7)

    day_of_year = int((month_number - 1) * 30.4 + 15)

    weather_input = pd.DataFrame([{
        "month": weather_month,
        "season": weather_season,
        "state": weather_state,

        "avg_temp": temperature,
        "min_temp": temperature - 3,
        "max_temp": temperature + 3,
        "wind_speed": wind_speed,
        "air_pressure": air_pressure,

        "elevation": 500.0,
        "latitude": 17.38,
        "longitude": 78.48,

        "previous_rainfall": rainfall,
        "previous_avg_temp": temperature,
        "previous_wind_speed": wind_speed,
        "previous_pressure": air_pressure,

        "rainfall_3day": rainfall,
        "rainfall_7day": rainfall,
        "rainfall_3day_avg": rainfall / 3,
        "rainfall_7day_avg": rainfall / 7,

        "temp_3day_avg": temperature,
        "temp_7day_avg": temperature,

        "pressure_change": 0.0,
        "temp_change": 0.0,
        "wind_change": 0.0,

        "day_of_year": day_of_year,

        "month_sin": np.sin(2 * np.pi * month_number / 12),
        "month_cos": np.cos(2 * np.pi * month_number / 12)
    }])

    weather_prediction = weather_model.predict(weather_input)[0]

    weather_labels = {
        0: "Extreme",
        1: "High",
        2: "Low",
        3: "Moderate"
    }

    weather_risk = weather_labels.get(
        int(weather_prediction),
        str(weather_prediction)
    )


    # --------------------------------------------------------
    # 4. RL IRRIGATION OPTIMIZATION
    # --------------------------------------------------------

    rl_state = np.array([
        soil_moisture / 1000,
        soil_temperature / 50,
        temperature / 50,
        humidity / 100,
        rainfall / 1000,
        wind_speed / 50
    ], dtype=np.float32)

    rl_state = np.clip(rl_state, 0, 1)

    action, _ = rl_model.predict(
        rl_state,
        deterministic=True
    )

    irrigation = (
        "IRRIGATION"
        if int(action) == 1
        else "NO IRRIGATION"
    )


    # ========================================================
    # 5. MARKET PRICE FORECASTING
    # ========================================================

    market_price = None

    try:

        # Convert month name to month number
        market_month_map = {
            "January": 1,
            "February": 2,
            "March": 3,
            "April": 4,
            "May": 5,
            "June": 6,
            "July": 7,
            "August": 8,
            "September": 9,
            "October": 10,
            "November": 11,
            "December": 12
        }

        market_month_number = market_month_map.get(
            str(month),
            1
        )

        # Approximate middle of selected month
        days_before_month = [
            0, 31, 59, 90, 120, 151,
            181, 212, 243, 273, 304, 334
        ]

        market_day_of_year = (
            days_before_month[market_month_number - 1] + 15
        )

        # IMPORTANT:
        # Market Price XGBoost model expects EXACTLY 8 features:
        # res_lag_1
        # res_lag_7
        # res_lag_14
        # res_lag_30
        # res_rolling_7
        # res_rolling_14
        # month
        # day_of_year

        market_features = np.array([[
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            float(market_month_number),
            float(market_day_of_year)
        ]])

        market_price = float(
            market_price_xgb_model.predict(
                market_features
            )[0]
        )

    except Exception as e:

        print(
            "Market Price Forecast Error:",
            e
        )

        market_price = None


    # --------------------------------------------------------
    # 6. CLIMATE CHANGE SCENARIO ANALYSIS
    # --------------------------------------------------------

    climate_result = {}

    try:

        base_input = pd.DataFrame([{
            "Crop": recommended_crop,
            "Crop_Year": crop_year,
            "Season": season,
            "State": state,
            "Area": area,
            "Annual_Rainfall": rainfall,
            "Fertilizer": fertilizer,
            "Pesticide": pesticide
        }])

        baseline_yield = float(
            climate_model.predict(base_input)[0]
        )

        moderate_input = base_input.copy()
        moderate_input["Annual_Rainfall"] = rainfall * 0.90

        severe_input = base_input.copy()
        severe_input["Annual_Rainfall"] = rainfall * 0.75

        moderate_yield = float(
            climate_model.predict(moderate_input)[0]
        )

        severe_yield = float(
            climate_model.predict(severe_input)[0]
        )

        climate_result = {
            "Baseline": baseline_yield,
            "Moderate Climate Change": moderate_yield,
            "Severe Climate Change": severe_yield
        }

    except Exception:

        climate_result = {
            "Baseline": float(predicted_yield),
            "Moderate Climate Change": float(predicted_yield),
            "Severe Climate Change": float(predicted_yield)
        }


    # --------------------------------------------------------
    # 7. SUSTAINABILITY SCORE
    # --------------------------------------------------------

    try:

        fertilizer_per_ha = fertilizer / max(area, 1)
        pesticide_per_ha = pesticide / max(area, 1)

        production_per_ha = predicted_yield

        sustainability_score = (
            50
            - 0.05 * fertilizer_per_ha
            - 2.0 * pesticide_per_ha
            + 2.0 * np.log1p(
                max(production_per_ha, 0)
            )
        )

        sustainability_score = float(
            np.clip(
                sustainability_score,
                0,
                100
            )
        )

        if sustainability_score >= 60:
            sustainability_category = "High Sustainability"

        elif sustainability_score >= 30:
            sustainability_category = "Moderate Sustainability"

        else:
            sustainability_category = "Low Sustainability"

    except Exception:

        sustainability_score = 0.0
        sustainability_category = "Low Sustainability"


    # --------------------------------------------------------
    # FINAL RESULT
    # --------------------------------------------------------

    return {
        "Recommended Crop": recommended_crop,

        "Predicted Yield": round(
            float(predicted_yield),
            4
        ),

        "Weather Risk": weather_risk,

        "Irrigation Decision": irrigation,

        "Market Price Forecast": (
            round(market_price, 2)
            if market_price is not None
            else "Unavailable"
        ),

        "Climate Change Scenarios": climate_result,

        "Sustainability Score": round(
            sustainability_score,
            2
        ),

        "Sustainability Category": sustainability_category
    }


print(
    "Corrected unified 7-module prediction function created successfully!"
)

In [ ]:
# ============================================================
# TEST THE COMPLETE 7-MODULE AGRICULTURE AI SYSTEM
# ============================================================

test_result = agriculture_ai_7_module_prediction(
    N=90,
    P=42,
    K=43,
    temperature=25.0,
    humidity=80.0,
    ph=6.5,
    rainfall=200.0,

    soil_moisture=200.0,
    soil_temperature=25.0,

    crop_year=2024,
    area=1000.0,
    fertilizer=50000.0,
    pesticide=500.0,

    season="Kharif",
    state="Andhra Pradesh",
    month="July",

    wind_speed=10.0,
    air_pressure=1010.0
)

print("==============================================")
print("   AGRICULTURE AI — 7 MODULE TEST")
print("==============================================")

for key, value in test_result.items():
    print(f"{key}: {value}")

print("==============================================")

In [ ]:
# ============================================================
# FINAL AGRICULTURE AI - 7 MODULE GRADIO DASHBOARD
# ============================================================

import gradio as gr

def run_agriculture_ai(
    N, P, K,
    temperature, humidity, ph, rainfall,
    soil_moisture, soil_temperature,
    crop_year, area, fertilizer, pesticide,
    season, state, month,
    wind_speed, air_pressure
):

    result = agriculture_ai_7_module_prediction(
        N=float(N),
        P=float(P),
        K=float(K),
        temperature=float(temperature),
        humidity=float(humidity),
        ph=float(ph),
        rainfall=float(rainfall),

        soil_moisture=float(soil_moisture),
        soil_temperature=float(soil_temperature),

        crop_year=int(crop_year),
        area=float(area),
        fertilizer=float(fertilizer),
        pesticide=float(pesticide),

        season=str(season),
        state=str(state),
        month=str(month),

        wind_speed=float(wind_speed),
        air_pressure=float(air_pressure)
    )

    return (
        result["Recommended Crop"],
        result["Predicted Yield"],
        result["Weather Risk"],
        result["Irrigation Decision"],
        result["Market Price Forecast"],
        str(result["Climate Change Scenarios"]),
        result["Sustainability Score"],
        result["Sustainability Category"]
    )


# ============================================================
# GRADIO INTERFACE
# ============================================================

with gr.Blocks(title="Agriculture AI - 7 Module System") as demo:

    gr.Markdown(
        """
        # 🌾 Agriculture AI - Integrated 7 Module System

        ### AI-powered agricultural decision support system

        This system combines:
        1. 🌱 Crop Recommendation
        2. 📈 Crop Yield Prediction
        3. 🌦️ Weather Risk Prediction
        4. 💧 RL Irrigation Optimization
        5. 💰 Market Price Forecasting
        6. 🌍 Climate Change Scenario Analysis
        7. ♻️ Carbon Footprint / Sustainability Score
        """
    )

    gr.Markdown("## 🌱 Agricultural Inputs")

    with gr.Row():
        N = gr.Number(label="Nitrogen (N)", value=90)
        P = gr.Number(label="Phosphorus (P)", value=42)
        K = gr.Number(label="Potassium (K)", value=43)

    with gr.Row():
        temperature = gr.Number(label="Temperature (°C)", value=25)
        humidity = gr.Number(label="Humidity (%)", value=80)
        ph = gr.Number(label="Soil pH", value=6.5)
        rainfall = gr.Number(label="Rainfall (mm)", value=200)

    gr.Markdown("## 💧 Soil & Irrigation Inputs")

    with gr.Row():
        soil_moisture = gr.Number(
            label="Soil Moisture",
            value=200
        )
        soil_temperature = gr.Number(
            label="Soil Temperature (°C)",
            value=25
        )

    gr.Markdown("## 🚜 Farm Information")

    with gr.Row():
        crop_year = gr.Number(
            label="Crop Year",
            value=2024,
            precision=0
        )
        area = gr.Number(
            label="Area",
            value=1000
        )

    with gr.Row():
        fertilizer = gr.Number(
            label="Fertilizer",
            value=50000
        )
        pesticide = gr.Number(
            label="Pesticide",
            value=500
        )

    gr.Markdown("## 🌦️ Seasonal & Weather Information")

    with gr.Row():
        season = gr.Dropdown(
            choices=[
                "Kharif",
                "Rabi",
                "Summer",
                "Autumn",
                "Whole Year",
                "Monsoon",
                "Post-monsoon",
                "Winter"
            ],
            value="Kharif",
            label="Season"
        )

        state = gr.Dropdown(
            choices=[
                "Andhra Pradesh",
                "Telangana",
                "Tamil Nadu",
                "Karnataka",
                "Maharashtra",
                "Assam",
                "West Bengal",
                "Punjab",
                "Haryana",
                "Uttar Pradesh"
            ],
            value="Andhra Pradesh",
            label="State"
        )

        month = gr.Dropdown(
            choices=[
                "January",
                "February",
                "March",
                "April",
                "May",
                "June",
                "July",
                "August",
                "September",
                "October",
                "November",
                "December"
            ],
            value="July",
            label="Month"
        )

    with gr.Row():
        wind_speed = gr.Number(
            label="Wind Speed",
            value=10
        )
        air_pressure = gr.Number(
            label="Air Pressure",
            value=1010
        )

    predict_button = gr.Button(
        "🚀 RUN AGRICULTURE AI",
        variant="primary"
    )

    gr.Markdown("## 📊 AI Predictions")

    with gr.Row():
        recommended_crop = gr.Textbox(
            label="🌱 Recommended Crop"
        )
        predicted_yield = gr.Number(
            label="📈 Predicted Yield"
        )

    with gr.Row():
        weather_risk = gr.Textbox(
            label="🌦️ Weather Risk"
        )
        irrigation = gr.Textbox(
            label="💧 Irrigation Decision"
        )

    with gr.Row():
        market_price = gr.Number(
            label="💰 Market Price Forecast"
        )
        sustainability_score = gr.Number(
            label="♻️ Sustainability Score"
        )

    sustainability_category = gr.Textbox(
        label="♻️ Sustainability Category"
    )

    climate_scenarios = gr.Textbox(
        label="🌍 Climate Change Scenarios",
        lines=4
    )

    predict_button.click(
        fn=run_agriculture_ai,
        inputs=[
            N, P, K,
            temperature, humidity, ph, rainfall,
            soil_moisture, soil_temperature,
            crop_year, area, fertilizer, pesticide,
            season, state, month,
            wind_speed, air_pressure
        ],
        outputs=[
            recommended_crop,
            predicted_yield,
            weather_risk,
            irrigation,
            market_price,
            climate_scenarios,
            sustainability_score,
            sustainability_category
        ]
    )

    gr.Markdown(
        """
        ---
        **Agriculture AI | 7-Module Integrated Decision Support System**
        """
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    share=True,
    debug=True
)

In [ ]:

# ============================================================
# XAI LIBRARIES
# ============================================================

!pip install -q shap

In [ ]:
# ============================================================
# XAI IMPORTS
# ============================================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("SHAP version:", shap.__version__)
print("XAI libraries loaded successfully!")

In [ ]:
# ============================================================
# CHECK ALL 7 MODELS
# ============================================================

print("1. Crop Model   :", type(crop_model))
print("2. Yield Model  :", type(yield_model))
print("3. Weather Model:", type(weather_model))
print("4. RL Model     :", type(rl_model))
print("5. Market Model :", type(market_price_xgb_model))

try:
    print("6. Climate Model:", type(climate_model))
except:
    print("6. Climate Model: Not directly available")

print("7. Sustainability: Formula-based")

print("\nAll existing models will be preserved.")

In [ ]:
# ============================================================
# CROP RECOMMENDATION - XAI SETUP
# ============================================================

crop_features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

print("Crop features:")
for i, feature in enumerate(crop_features, 1):
    print(i, "->", feature)

In [ ]:
# ============================================================
# CROP RECOMMENDATION - SHAP EXPLAINER
# ============================================================

crop_xai_model = crop_model

# Handle Pipeline models
if hasattr(crop_model, "named_steps"):
    print("Crop model is a Pipeline.")

    print("Pipeline steps:")
    for name, step in crop_model.named_steps.items():
        print(" -", name, ":", type(step))

else:
    print("Crop model is a direct estimator.")

print("Crop XAI setup ready.")

In [ ]:
# ============================================================
# CROP RECOMMENDATION - INPUT SENSITIVITY XAI
# ============================================================

def explain_crop_prediction(N, P, K, temperature, humidity, ph, rainfall):

    base_input = pd.DataFrame([[
        N, P, K, temperature, humidity, ph, rainfall
    ]], columns=crop_features)

    base_prediction = crop_model.predict(base_input)[0]

    input_values = {
        "Nitrogen (N)": N,
        "Phosphorus (P)": P,
        "Potassium (K)": K,
        "Temperature (°C)": temperature,
        "Humidity (%)": humidity,
        "pH": ph,
        "Rainfall (mm)": rainfall
    }

    feature_columns = [
        "N", "P", "K", "temperature",
        "humidity", "ph", "rainfall"
    ]

    explanations = []

    for feature_name, column_name in zip(input_values.keys(), feature_columns):

        original_value = base_input[column_name].iloc[0]

        # Change the feature by approximately 10%
        if original_value == 0:
            changed_value = 1
        else:
            changed_value = original_value * 1.10

        modified_input = base_input.copy()
        modified_input[column_name] = changed_value

        modified_prediction = crop_model.predict(modified_input)[0]

        if modified_prediction == base_prediction:
            effect = "supports the same crop prediction"
        else:
            effect = f"can shift the prediction toward {modified_prediction}"

        explanations.append(
            f"{feature_name} = {original_value:.2f}: "
            f"when increased by 10%, the model {effect}."
        )

    return {
        "prediction": base_prediction,
        "explanations": explanations
    }


print("Crop XAI function created successfully.")

In [ ]:
# ============================================================
# TEST CROP XAI
# ============================================================

crop_xai_result = explain_crop_prediction(
    N=90,
    P=42,
    K=43,
    temperature=25,
    humidity=80,
    ph=6.5,
    rainfall=200
)

print("==========================================")
print("CROP XAI RESULT")
print("==========================================")

print("Recommended Crop:", crop_xai_result["prediction"])

print("\nExplanation:")
for explanation in crop_xai_result["explanations"]:
    print("-", explanation)

In [ ]:
# ============================================================
# CROP YIELD - INPUT SENSITIVITY XAI
# ============================================================

yield_features = [
    "Crop",
    "Crop_Year",
    "Season",
    "State",
    "Area",
    "Annual_Rainfall",
    "Fertilizer",
    "Pesticide"
]

print("Yield XAI features:")
for i, feature in enumerate(yield_features, 1):
    print(i, "->", feature)

print("\nYield XAI setup ready.")

In [ ]:
# ============================================================
# CROP YIELD - INPUT SENSITIVITY XAI FUNCTION
# ============================================================

def explain_yield_prediction(
    crop,
    crop_year,
    season,
    state,
    area,
    annual_rainfall,
    fertilizer,
    pesticide
):

    base_input = pd.DataFrame([[
        crop,
        crop_year,
        season,
        state,
        area,
        annual_rainfall,
        fertilizer,
        pesticide
    ]], columns=yield_features)

    base_prediction = float(yield_model.predict(base_input)[0])

    # User-entered values
    input_values = {
        "Crop": crop,
        "Crop Year": crop_year,
        "Season": season,
        "State": state,
        "Area": area,
        "Annual Rainfall": annual_rainfall,
        "Fertilizer": fertilizer,
        "Pesticide": pesticide
    }

    explanations = []

    # Numerical features
    numerical_changes = {
        "Crop Year": "Crop_Year",
        "Area": "Area",
        "Annual Rainfall": "Annual_Rainfall",
        "Fertilizer": "Fertilizer",
        "Pesticide": "Pesticide"
    }

    for feature_name, column_name in numerical_changes.items():

        original_value = base_input[column_name].iloc[0]

        # Increase numerical input by 10%
        if original_value == 0:
            changed_value = 1
        else:
            changed_value = original_value * 1.10

        modified_input = base_input.copy()
        modified_input[column_name] = changed_value

        modified_prediction = float(
            yield_model.predict(modified_input)[0]
        )

        difference = modified_prediction - base_prediction

        if difference > 0:
            effect = "increases"
        elif difference < 0:
            effect = "decreases"
        else:
            effect = "does not change"

        explanations.append(
            f"{feature_name} = {original_value}: "
            f"a 10% increase {effect} the predicted yield "
            f"by {abs(difference):.4f}."
        )

    # Categorical features
    categorical_features = {
        "Crop": "Crop",
        "Season": "Season",
        "State": "State"
    }

    for feature_name, column_name in categorical_features.items():

        original_value = base_input[column_name].iloc[0]

        # Keep the original category because there is no
        # safe universal alternative category to substitute.
        explanations.append(
            f"{feature_name} = {original_value}: "
            f"this categorical input is used directly by the yield model."
        )

    return {
        "prediction": base_prediction,
        "explanations": explanations
    }


print("Yield XAI function created successfully.")

In [ ]:
# ============================================================
# TEST YIELD XAI
# ============================================================

yield_xai_result = explain_yield_prediction(
    crop="rice",
    crop_year=2025,
    season="Kharif",
    state="Andhra Pradesh",
    area=100,
    annual_rainfall=200,
    fertilizer=50000,
    pesticide=500
)

print("==========================================")
print("CROP YIELD XAI RESULT")
print("==========================================")

print("Predicted Yield:", round(yield_xai_result["prediction"], 4))

print("\nExplanation:")
for explanation in yield_xai_result["explanations"]:
    print("-", explanation)

In [ ]:
# ============================================================
# WEATHER RISK - XAI SETUP
# ============================================================

weather_xai_features = [
    "avg_temp",
    "min_temp",
    "max_temp",
    "wind_speed",
    "air_pressure",
    "previous_rainfall",
    "previous_avg_temp",
    "previous_wind_speed",
    "previous_pressure",
    "rainfall_3day",
    "rainfall_7day",
    "rainfall_3day_avg",
    "rainfall_7day_avg",
    "temp_3day_avg",
    "temp_7day_avg",
    "pressure_change",
    "temp_change",
    "wind_change"
]

print("Weather XAI setup ready.")
print("User inputs will be connected to the model's derived weather features.")
print("Number of tracked weather features:", len(weather_xai_features))

In [ ]:
# ============================================================
# WEATHER RISK - CORRECTED XAI FUNCTION
# ============================================================

def explain_weather_prediction(
    temperature,
    humidity,
    rainfall,
    wind_speed,
    air_pressure,
    state="Andhra Pradesh",
    season="Kharif",
    month="July"
):

    # --------------------------------------------------------
    # SAME MAPPINGS USED BY THE MAIN AGRICULTURE SYSTEM
    # --------------------------------------------------------

    state_codes = {
        "Andhra Pradesh": "AP",
        "Arunachal Pradesh": "AR",
        "Assam": "AS",
        "Bihar": "BR",
        "Chhattisgarh": "CT",
        "Goa": "GA",
        "Gujarat": "GJ",
        "Haryana": "HR",
        "Himachal Pradesh": "HP",
        "Jharkhand": "JH",
        "Karnataka": "KA",
        "Kerala": "KL",
        "Madhya Pradesh": "MP",
        "Maharashtra": "MH",
        "Manipur": "MN",
        "Meghalaya": "ML",
        "Mizoram": "MZ",
        "Nagaland": "NL",
        "Odisha": "OR",
        "Punjab": "PB",
        "Rajasthan": "RJ",
        "Sikkim": "SK",
        "Tamil Nadu": "TN",
        "Telangana": "TG",
        "Tripura": "TR",
        "Uttar Pradesh": "UP",
        "Uttarakhand": "UK",
        "West Bengal": "WB",
        "Delhi": "DL",
        "Jammu and Kashmir": "JK",
        "Puducherry": "PY",
        "Andaman and Nicobar Islands": "AN"
    }

    season_map = {
        "Kharif": "Monsoon",
        "Rabi": "Winter",
        "Whole Year": "Summer",
        "Autumn": "Post-monsoon",
        "Summer": "Summer",
        "Winter": "Winter",
        "Monsoon": "Monsoon",
        "Post-monsoon": "Post-monsoon"
    }

    month_names = {
        1: "January",
        2: "February",
        3: "March",
        4: "April",
        5: "May",
        6: "June",
        7: "July",
        8: "August",
        9: "September",
        10: "October",
        11: "November",
        12: "December"
    }

    weather_state = state_codes.get(state, state)
    weather_season = season_map.get(season, season)

    if isinstance(month, (int, np.integer)):
        weather_month = month_names.get(int(month), "July")
    else:
        weather_month = str(month)

    month_number = {
        "January": 1,
        "February": 2,
        "March": 3,
        "April": 4,
        "May": 5,
        "June": 6,
        "July": 7,
        "August": 8,
        "September": 9,
        "October": 10,
        "November": 11,
        "December": 12
    }[weather_month]

    # --------------------------------------------------------
    # FUNCTION TO BUILD THE EXACT WEATHER INPUT
    # --------------------------------------------------------

    def build_weather_input(
        temp,
        hum,
        rain,
        wind,
        pressure
    ):

        day_of_year = int((month_number - 1) * 30.4 + 15)

        return pd.DataFrame([{
            "month": weather_month,
            "season": weather_season,
            "state": weather_state,

            "avg_temp": temp,
            "min_temp": temp - 3,
            "max_temp": temp + 3,

            "wind_speed": wind,
            "air_pressure": pressure,

            "elevation": 500.0,
            "latitude": 17.38,
            "longitude": 78.48,

            "previous_rainfall": rain,
            "previous_avg_temp": temp,
            "previous_wind_speed": wind,
            "previous_pressure": pressure,

            "rainfall_3day": rain,
            "rainfall_7day": rain,

            "rainfall_3day_avg": rain / 3,
            "rainfall_7day_avg": rain / 7,

            "temp_3day_avg": temp,
            "temp_7day_avg": temp,

            "pressure_change": 0.0,
            "temp_change": 0.0,
            "wind_change": 0.0,

            "day_of_year": day_of_year,

            "month_sin": np.sin(
                2 * np.pi * month_number / 12
            ),

            "month_cos": np.cos(
                2 * np.pi * month_number / 12
            )
        }])

    # --------------------------------------------------------
    # BASE PREDICTION
    # --------------------------------------------------------

    base_input = build_weather_input(
        temperature,
        humidity,
        rainfall,
        wind_speed,
        air_pressure
    )

    base_prediction = int(
        weather_model.predict(base_input)[0]
    )

    risk_mapping = {
        0: "Extreme",
        1: "High",
        2: "Low",
        3: "Moderate"
    }

    base_risk = risk_mapping.get(
        base_prediction,
        str(base_prediction)
    )

    # --------------------------------------------------------
    # INPUT SENSITIVITY
    # --------------------------------------------------------

    user_inputs = {
        "Temperature (°C)": "temperature",
        "Humidity (%)": "humidity",
        "Rainfall (mm)": "rainfall",
        "Wind Speed": "wind_speed",
        "Air Pressure": "air_pressure"
    }

    explanations = []

    for feature_name, feature_key in user_inputs.items():

        modified_temperature = temperature
        modified_humidity = humidity
        modified_rainfall = rainfall
        modified_wind = wind_speed
        modified_pressure = air_pressure

        if feature_key == "temperature":
            modified_temperature *= 1.10

        elif feature_key == "humidity":
            modified_humidity *= 1.10

        elif feature_key == "rainfall":
            modified_rainfall *= 1.10

        elif feature_key == "wind_speed":
            modified_wind *= 1.10

        elif feature_key == "air_pressure":
            modified_pressure *= 1.10

        modified_input = build_weather_input(
            modified_temperature,
            modified_humidity,
            modified_rainfall,
            modified_wind,
            modified_pressure
        )

        modified_prediction = int(
            weather_model.predict(modified_input)[0]
        )

        modified_risk = risk_mapping.get(
            modified_prediction,
            str(modified_prediction)
        )

        original_value = {
            "temperature": temperature,
            "humidity": humidity,
            "rainfall": rainfall,
            "wind_speed": wind_speed,
            "air_pressure": air_pressure
        }[feature_key]

        if modified_risk == base_risk:
            effect = "does not change the predicted risk"
        else:
            effect = (
                f"changes the predicted risk "
                f"from {base_risk} to {modified_risk}"
            )

        explanations.append(
            f"{feature_name} = {original_value:.2f}: "
            f"a 10% increase {effect}."
        )

    return {
        "prediction": base_risk,
        "explanations": explanations
    }


print("Corrected Weather Risk XAI function created successfully.")

In [ ]:
# ============================================================
# WEATHER MODEL - INSPECT SAVED MODEL
# ============================================================

print("Weather model type:")
print(type(weather_model))

print("\nModel attributes:")

if hasattr(weather_model, "named_steps"):
    print("This is a Pipeline.")
    for name, step in weather_model.named_steps.items():
        print("-", name, ":", type(step))
else:
    print("This is a direct estimator.")

print("\nFeature names:")
if hasattr(weather_model, "feature_names_in_"):
    print(weather_model.feature_names_in_)
else:
    print("feature_names_in_ not available")

print("\nWeather model inspection complete.")

In [ ]:
# ============================================================
# WEATHER MODEL - INSPECT PREPROCESSOR
# ============================================================

preprocessor = weather_model.named_steps["preprocessor"]

print("Weather preprocessing structure:")
print(preprocessor)

print("\nTransformers:")

for name, transformer, columns in preprocessor.transformers_:
    print("\nTransformer:", name)
    print("Type:", type(transformer))
    print("Columns:", columns)

    if hasattr(transformer, "categories_"):
        print("Categories:")
        for i, categories in enumerate(transformer.categories_):
            print(f"  Column {i}:", categories[:20])

print("\nPreprocessor inspection complete.")

In [ ]:
# ============================================================
# TEST CORRECTED WEATHER XAI
# ============================================================

weather_xai_result = explain_weather_prediction(
    temperature=25,
    humidity=80,
    rainfall=200,
    wind_speed=10,
    air_pressure=1010,
    state="Andhra Pradesh",
    season="Kharif",
    month="July"
)

print("==========================================")
print("WEATHER RISK XAI RESULT")
print("==========================================")

print(
    "Predicted Weather Risk:",
    weather_xai_result["prediction"]
)

print("\nExplanation:")

for explanation in weather_xai_result["explanations"]:
    print("-", explanation)

In [ ]:
# ============================================================
# IRRIGATION - INPUT SENSITIVITY XAI
# ============================================================

def explain_irrigation_prediction(
    soil_moisture,
    soil_temperature,
    temperature,
    humidity,
    rainfall,
    wind_speed
):

    def get_action(
        sm,
        st,
        temp,
        hum,
        rain,
        wind
    ):
        state = np.array([
            sm / 1000,
            st / 50,
            temp / 50,
            hum / 100,
            rain / 1000,
            wind / 50
        ], dtype=np.float32)

        state = np.clip(state, 0, 1)

        action, _ = rl_model.predict(
            state,
            deterministic=True
        )

        return int(action)

    # Base prediction
    base_action = get_action(
        soil_moisture,
        soil_temperature,
        temperature,
        humidity,
        rainfall,
        wind_speed
    )

    base_decision = (
        "IRRIGATION"
        if base_action == 1
        else "NO IRRIGATION"
    )

    user_inputs = {
        "Soil Moisture": soil_moisture,
        "Soil Temperature": soil_temperature,
        "Temperature": temperature,
        "Humidity": humidity,
        "Rainfall": rainfall,
        "Wind Speed": wind_speed
    }

    explanations = []

    for feature_name, original_value in user_inputs.items():

        values = {
            "soil_moisture": soil_moisture,
            "soil_temperature": soil_temperature,
            "temperature": temperature,
            "humidity": humidity,
            "rainfall": rainfall,
            "wind_speed": wind_speed
        }

        key = {
            "Soil Moisture": "soil_moisture",
            "Soil Temperature": "soil_temperature",
            "Temperature": "temperature",
            "Humidity": "humidity",
            "Rainfall": "rainfall",
            "Wind Speed": "wind_speed"
        }[feature_name]

        values[key] = (
            original_value * 1.10
            if original_value != 0
            else 1
        )

        modified_action = get_action(
            values["soil_moisture"],
            values["soil_temperature"],
            values["temperature"],
            values["humidity"],
            values["rainfall"],
            values["wind_speed"]
        )

        modified_decision = (
            "IRRIGATION"
            if modified_action == 1
            else "NO IRRIGATION"
        )

        if modified_decision == base_decision:
            effect = "does not change the irrigation decision"
        else:
            effect = (
                f"changes the decision from "
                f"{base_decision} to {modified_decision}"
            )

        explanations.append(
            f"{feature_name} = {original_value:.2f}: "
            f"a 10% increase {effect}."
        )

    return {
        "prediction": base_decision,
        "explanations": explanations
    }


print("Irrigation XAI function created successfully.")

In [ ]:
# ============================================================
# TEST IRRIGATION XAI
# ============================================================

irrigation_xai_result = explain_irrigation_prediction(
    soil_moisture=200,
    soil_temperature=25,
    temperature=25,
    humidity=80,
    rainfall=200,
    wind_speed=10
)

print("==========================================")
print("IRRIGATION XAI RESULT")
print("==========================================")

print(
    "Irrigation Decision:",
    irrigation_xai_result["prediction"]
)

print("\nExplanation:")

for explanation in irrigation_xai_result["explanations"]:
    print("-", explanation)

In [ ]:
# ============================================================
# MARKET PRICE - XAI SETUP
# ============================================================

market_xai_features = [
    "res_lag_1",
    "res_lag_7",
    "res_lag_14",
    "res_lag_30",
    "res_rolling_7",
    "res_rolling_14",
    "month",
    "day_of_year"
]

print("Market Price XAI features:")

for i, feature in enumerate(market_xai_features, 1):
    print(i, "->", feature)

print("\nMarket Price XAI setup ready.")

In [ ]:
# ============================================================
# MARKET PRICE - INPUT SENSITIVITY XAI
# ============================================================

def explain_market_price_prediction(
    month=7,
    day_of_year=182
):

    # --------------------------------------------------------
    # Current integrated system uses zero lag/rolling values
    # --------------------------------------------------------

    base_input = pd.DataFrame([[
        0.0,   # res_lag_1
        0.0,   # res_lag_7
        0.0,   # res_lag_14
        0.0,   # res_lag_30
        0.0,   # res_rolling_7
        0.0,   # res_rolling_14
        month,
        day_of_year
    ]], columns=market_xai_features)

    base_prediction = float(
        market_price_xgb_model.predict(base_input)[0]
    )

    explanations = []

    # --------------------------------------------------------
    # Month sensitivity
    # --------------------------------------------------------

    modified_month = min(month + 1, 12)

    month_input = base_input.copy()
    month_input["month"] = modified_month

    month_prediction = float(
        market_price_xgb_model.predict(month_input)[0]
    )

    month_difference = month_prediction - base_prediction

    if month_difference > 0:
        month_effect = "increases"
    elif month_difference < 0:
        month_effect = "decreases"
    else:
        month_effect = "does not change"

    explanations.append(
        f"Month = {month}: changing the month to "
        f"{modified_month} {month_effect} the forecast "
        f"by {abs(month_difference):.4f}."
    )

    # --------------------------------------------------------
    # Day-of-year sensitivity
    # --------------------------------------------------------

    modified_day = min(day_of_year + 30, 365)

    day_input = base_input.copy()
    day_input["day_of_year"] = modified_day

    day_prediction = float(
        market_price_xgb_model.predict(day_input)[0]
    )

    day_difference = day_prediction - base_prediction

    if day_difference > 0:
        day_effect = "increases"
    elif day_difference < 0:
        day_effect = "decreases"
    else:
        day_effect = "does not change"

    explanations.append(
        f"Day of year = {day_of_year}: changing it to "
        f"{modified_day} {day_effect} the forecast "
        f"by {abs(day_difference):.4f}."
    )

    # --------------------------------------------------------
    # Explain lag/rolling inputs used by current integration
    # --------------------------------------------------------

    explanations.append(
        "Lag features (1, 7, 14 and 30 days) are currently "
        "set to 0.0 in the integrated system because historical "
        "commodity-price observations are not entered by the user."
    )

    explanations.append(
        "Rolling 7-day and 14-day features are currently "
        "set to 0.0 for the same reason."
    )

    return {
        "prediction": base_prediction,
        "explanations": explanations
    }


print("Market Price XAI function created successfully.")

In [ ]:
# ============================================================
# TEST MARKET PRICE XAI
# ============================================================

market_xai_result = explain_market_price_prediction(
    month=7,
    day_of_year=182
)

print("==========================================")
print("MARKET PRICE XAI RESULT")
print("==========================================")

print(
    "Market Price Forecast:",
    round(market_xai_result["prediction"], 4)
)

print("\nExplanation:")

for explanation in market_xai_result["explanations"]:
    print("-", explanation)

In [ ]:
# ============================================================
# CLIMATE CHANGE - XAI
# ============================================================

def explain_climate_change(
    crop,
    crop_year,
    season,
    state,
    area,
    rainfall,
    fertilizer,
    pesticide
):

    # --------------------------------------------------------
    # Baseline scenario
    # --------------------------------------------------------

    baseline_input = pd.DataFrame([{
        "Crop": crop,
        "Crop_Year": crop_year,
        "Season": season,
        "State": state,
        "Area": area,
        "Annual_Rainfall": rainfall,
        "Fertilizer": fertilizer,
        "Pesticide": pesticide
    }])

    baseline_yield = float(
        yield_model.predict(baseline_input)[0]
    )

    # --------------------------------------------------------
    # Moderate climate scenario
    # 10% rainfall reduction
    # --------------------------------------------------------

    moderate_rainfall = rainfall * 0.90

    moderate_input = baseline_input.copy()
    moderate_input["Annual_Rainfall"] = moderate_rainfall

    moderate_yield = float(
        yield_model.predict(moderate_input)[0]
    )

    # --------------------------------------------------------
    # Severe climate scenario
    # 25% rainfall reduction
    # --------------------------------------------------------

    severe_rainfall = rainfall * 0.75

    severe_input = baseline_input.copy()
    severe_input["Annual_Rainfall"] = severe_rainfall

    severe_yield = float(
        yield_model.predict(severe_input)[0]
    )

    # --------------------------------------------------------
    # Calculate changes
    # --------------------------------------------------------

    moderate_change = moderate_yield - baseline_yield
    severe_change = severe_yield - baseline_yield

    return {
        "baseline": {
            "rainfall": rainfall,
            "yield": baseline_yield
        },

        "moderate": {
            "rainfall": moderate_rainfall,
            "yield": moderate_yield,
            "yield_change": moderate_change
        },

        "severe": {
            "rainfall": severe_rainfall,
            "yield": severe_yield,
            "yield_change": severe_change
        },

        "explanations": [
            f"Baseline rainfall = {rainfall:.2f} mm "
            f"with predicted yield = {baseline_yield:.4f}.",

            f"Moderate scenario reduces rainfall by 10% "
            f"to {moderate_rainfall:.2f} mm. "
            f"Predicted yield = {moderate_yield:.4f}.",

            f"Severe scenario reduces rainfall by 25% "
            f"to {severe_rainfall:.2f} mm. "
            f"Predicted yield = {severe_yield:.4f}."
        ]
    }


print("Climate Change XAI function created successfully.")

In [ ]:
# ============================================================
# TEST CLIMATE CHANGE XAI
# ============================================================

climate_xai_result = explain_climate_change(
    crop="rice",
    crop_year=2025,
    season="Kharif",
    state="Andhra Pradesh",
    area=100,
    rainfall=200,
    fertilizer=50000,
    pesticide=500
)

print("==========================================")
print("CLIMATE CHANGE XAI RESULT")
print("==========================================")

print(
    "Baseline Yield:",
    round(climate_xai_result["baseline"]["yield"], 4)
)

print(
    "Moderate Scenario Yield:",
    round(climate_xai_result["moderate"]["yield"], 4)
)

print(
    "Severe Scenario Yield:",
    round(climate_xai_result["severe"]["yield"], 4)
)

print("\nExplanation:")

for explanation in climate_xai_result["explanations"]:
    print("-", explanation)

In [ ]:
# ============================================================
# SUSTAINABILITY - XAI
# ============================================================

def explain_sustainability(
    area,
    fertilizer,
    pesticide,
    predicted_yield
):

    # --------------------------------------------------------
    # Calculate normalized inputs
    # --------------------------------------------------------

    fertilizer_per_ha = fertilizer / max(area, 1)
    pesticide_per_ha = pesticide / max(area, 1)
    production_per_ha = predicted_yield

    # --------------------------------------------------------
    # Individual formula contributions
    # Formula used in the integrated system:
    #
    # Score =
    # 50
    # - 0.05 * fertilizer_per_ha
    # - 2.0 * pesticide_per_ha
    # + 2.0 * log(1 + production_per_ha)
    # --------------------------------------------------------

    base_score = 50.0

    fertilizer_effect = -0.05 * fertilizer_per_ha
    pesticide_effect = -2.0 * pesticide_per_ha
    production_effect = (
        2.0 * np.log1p(max(production_per_ha, 0))
    )

    raw_score = (
        base_score
        + fertilizer_effect
        + pesticide_effect
        + production_effect
    )

    sustainability_score = float(
        np.clip(raw_score, 0, 100)
    )

    # --------------------------------------------------------
    # Category
    # --------------------------------------------------------

    if sustainability_score >= 60:
        category = "High Sustainability"
    elif sustainability_score >= 30:
        category = "Moderate Sustainability"
    else:
        category = "Low Sustainability"

    # --------------------------------------------------------
    # Detailed explanation
    # --------------------------------------------------------

    explanations = [
        f"Area = {area:.2f}: used to calculate "
        f"fertilizer and pesticide per hectare.",

        f"Fertilizer = {fertilizer:.2f}, "
        f"fertilizer per hectare = {fertilizer_per_ha:.2f}. "
        f"This contributes {fertilizer_effect:.4f} points "
        f"to the score.",

        f"Pesticide = {pesticide:.2f}, "
        f"pesticide per hectare = {pesticide_per_ha:.2f}. "
        f"This contributes {pesticide_effect:.4f} points "
        f"to the score.",

        f"Predicted production per hectare = "
        f"{production_per_ha:.4f}. "
        f"This contributes {production_effect:.4f} points "
        f"to the score.",

        f"Base sustainability score = {base_score:.2f}.",

        f"Final sustainability score = "
        f"{sustainability_score:.2f}.",

        f"Sustainability category = {category}."
    ]

    return {
        "score": sustainability_score,
        "category": category,
        "explanations": explanations
    }


print("Sustainability XAI function created successfully.")

In [ ]:
# ============================================================
# TEST SUSTAINABILITY XAI
# ============================================================

sustainability_xai_result = explain_sustainability(
    area=100,
    fertilizer=50000,
    pesticide=500,
    predicted_yield=0.772
)

print("==========================================")
print("SUSTAINABILITY XAI RESULT")
print("==========================================")

print(
    "Sustainability Score:",
    round(sustainability_xai_result["score"], 2)
)

print(
    "Sustainability Category:",
    sustainability_xai_result["category"]
)

print("\nExplanation:")

for explanation in sustainability_xai_result["explanations"]:
    print("-", explanation)

In [ ]:
# ============================================================
# COMBINED XAI FUNCTION
# ============================================================

def agriculture_ai_xai_prediction(
    N, P, K,
    temperature, humidity, ph, rainfall,
    soil_moisture, soil_temperature,
    crop_year, area, fertilizer, pesticide,
    season, state, month,
    wind_speed, air_pressure
):

    # --------------------------------------------------------
    # 1. CROP XAI
    # --------------------------------------------------------

    crop_result = explain_crop_prediction(
        N=N,
        P=P,
        K=K,
        temperature=temperature,
        humidity=humidity,
        ph=ph,
        rainfall=rainfall
    )

    recommended_crop = crop_result["prediction"]

    # --------------------------------------------------------
    # 2. YIELD XAI
    # --------------------------------------------------------

    yield_result = explain_yield_prediction(
        crop=recommended_crop,
        crop_year=crop_year,
        season=season,
        state=state,
        area=area,
        annual_rainfall=rainfall,
        fertilizer=fertilizer,
        pesticide=pesticide
    )

    predicted_yield = yield_result["prediction"]

    # --------------------------------------------------------
    # 3. WEATHER XAI
    # --------------------------------------------------------

    weather_result = explain_weather_prediction(
        temperature=temperature,
        humidity=humidity,
        rainfall=rainfall,
        wind_speed=wind_speed,
        air_pressure=air_pressure,
        state=state,
        season=season,
        month=month
    )

    # --------------------------------------------------------
    # 4. IRRIGATION XAI
    # --------------------------------------------------------

    irrigation_result = explain_irrigation_prediction(
        soil_moisture=soil_moisture,
        soil_temperature=soil_temperature,
        temperature=temperature,
        humidity=humidity,
        rainfall=rainfall,
        wind_speed=wind_speed
    )

    # --------------------------------------------------------
    # 5. MARKET PRICE XAI
    # --------------------------------------------------------

    month_names = {
        1: "January",
        2: "February",
        3: "March",
        4: "April",
        5: "May",
        6: "June",
        7: "July",
        8: "August",
        9: "September",
        10: "October",
        11: "November",
        12: "December"
    }

    if isinstance(month, str):
        month_number = {
            "January": 1,
            "February": 2,
            "March": 3,
            "April": 4,
            "May": 5,
            "June": 6,
            "July": 7,
            "August": 8,
            "September": 9,
            "October": 10,
            "November": 11,
            "December": 12
        }.get(month, 7)
    else:
        month_number = int(month)

    day_of_year = int(
        (month_number - 1) * 30.4 + 15
    )

    market_result = explain_market_price_prediction(
        month=month_number,
        day_of_year=day_of_year
    )

    # --------------------------------------------------------
    # 6. CLIMATE XAI
    # --------------------------------------------------------

    climate_result = explain_climate_change(
        crop=recommended_crop,
        crop_year=crop_year,
        season=season,
        state=state,
        area=area,
        rainfall=rainfall,
        fertilizer=fertilizer,
        pesticide=pesticide
    )

    # --------------------------------------------------------
    # 7. SUSTAINABILITY XAI
    # --------------------------------------------------------

    sustainability_result = explain_sustainability(
        area=area,
        fertilizer=fertilizer,
        pesticide=pesticide,
        predicted_yield=predicted_yield
    )

    # --------------------------------------------------------
    # RETURN ALL RESULTS
    # --------------------------------------------------------

    return {
        "Recommended Crop": recommended_crop,
        "Crop Explanation": crop_result["explanations"],

        "Predicted Yield": round(predicted_yield, 4),
        "Yield Explanation": yield_result["explanations"],

        "Weather Risk": weather_result["prediction"],
        "Weather Explanation": weather_result["explanations"],

        "Irrigation Decision": irrigation_result["prediction"],
        "Irrigation Explanation": irrigation_result["explanations"],

        "Market Price Forecast": round(
            market_result["prediction"], 4
        ),
        "Market Price Explanation": market_result["explanations"],

        "Climate Change Scenarios": climate_result,

        "Sustainability Score": round(
            sustainability_result["score"], 2
        ),
        "Sustainability Category": sustainability_result["category"],
        "Sustainability Explanation": sustainability_result["explanations"]
    }


print("Combined 7-module XAI function created successfully.")

In [ ]:
# ============================================================
# TEST COMPLETE 7-MODULE XAI
# ============================================================

xai_result = agriculture_ai_xai_prediction(
    N=90,
    P=42,
    K=43,
    temperature=25,
    humidity=80,
    ph=6.5,
    rainfall=200,
    soil_moisture=200,
    soil_temperature=25,
    crop_year=2025,
    area=100,
    fertilizer=50000,
    pesticide=500,
    season="Kharif",
    state="Andhra Pradesh",
    month="July",
    wind_speed=10,
    air_pressure=1010
)

print("==============================================")
print("COMPLETE 7-MODULE AGRICULTURE AI XAI")
print("==============================================")

print("\n1. CROP")
print("Prediction:", xai_result["Recommended Crop"])
for x in xai_result["Crop Explanation"]:
    print("-", x)

print("\n2. YIELD")
print("Prediction:", xai_result["Predicted Yield"])
for x in xai_result["Yield Explanation"]:
    print("-", x)

print("\n3. WEATHER")
print("Risk:", xai_result["Weather Risk"])
for x in xai_result["Weather Explanation"]:
    print("-", x)

print("\n4. IRRIGATION")
print("Decision:", xai_result["Irrigation Decision"])
for x in xai_result["Irrigation Explanation"]:
    print("-", x)

print("\n5. MARKET PRICE")
print("Forecast:", xai_result["Market Price Forecast"])
for x in xai_result["Market Price Explanation"]:
    print("-", x)

print("\n6. CLIMATE CHANGE")
for x in xai_result["Climate Change Scenarios"]["explanations"]:
    print("-", x)

print("\n7. SUSTAINABILITY")
print("Score:", xai_result["Sustainability Score"])
print("Category:", xai_result["Sustainability Category"])
for x in xai_result["Sustainability Explanation"]:
    print("-", x)

print("\n==============================================")
print("ALL 7 XAI MODULES COMPLETED")
print("==============================================")

In [ ]:
# ============================================================
# GRADIO XAI WRAPPER
# ============================================================

def gradio_xai_prediction(
    N, P, K,
    temperature, humidity, ph, rainfall,
    soil_moisture, soil_temperature,
    crop_year, area, fertilizer, pesticide,
    season, state, month,
    wind_speed, air_pressure
):

    result = agriculture_ai_xai_prediction(
        N, P, K,
        temperature, humidity, ph, rainfall,
        soil_moisture, soil_temperature,
        crop_year, area, fertilizer, pesticide,
        season, state, month,
        wind_speed, air_pressure
    )

    # --------------------------------------------------------
    # Basic prediction summary
    # --------------------------------------------------------

    summary = f"""
🌾 CROP RECOMMENDATION
Recommended Crop: {result['Recommended Crop']}

📈 CROP YIELD
Predicted Yield: {result['Predicted Yield']}

🌦️ WEATHER RISK
Risk: {result['Weather Risk']}

💧 IRRIGATION
Decision: {result['Irrigation Decision']}

💰 MARKET PRICE
Forecast: {result['Market Price Forecast']}

🌍 CLIMATE CHANGE
Baseline Yield: {result['Climate Change Scenarios']['baseline']['yield']:.4f}
Moderate Scenario Yield: {result['Climate Change Scenarios']['moderate']['yield']:.4f}
Severe Scenario Yield: {result['Climate Change Scenarios']['severe']['yield']:.4f}

♻️ SUSTAINABILITY
Score: {result['Sustainability Score']}
Category: {result['Sustainability Category']}
"""

    # --------------------------------------------------------
    # Detailed XAI explanation
    # --------------------------------------------------------

    explanation = "\n\n".join([
        "🌾 CROP RECOMMENDATION EXPLANATION\n" +
        "\n".join(
            "• " + x
            for x in result["Crop Explanation"]
        ),

        "📈 YIELD EXPLANATION\n" +
        "\n".join(
            "• " + x
            for x in result["Yield Explanation"]
        ),

        "🌦️ WEATHER RISK EXPLANATION\n" +
        "\n".join(
            "• " + x
            for x in result["Weather Explanation"]
        ),

        "💧 IRRIGATION EXPLANATION\n" +
        "\n".join(
            "• " + x
            for x in result["Irrigation Explanation"]
        ),

        "💰 MARKET PRICE EXPLANATION\n" +
        "\n".join(
            "• " + x
            for x in result["Market Price Explanation"]
        ),

        "🌍 CLIMATE CHANGE EXPLANATION\n" +
        "\n".join(
            "• " + x
            for x in result["Climate Change Scenarios"]["explanations"]
        ),

        "♻️ SUSTAINABILITY EXPLANATION\n" +
        "\n".join(
            "• " + x
            for x in result["Sustainability Explanation"]
        )
    ])

    return summary, explanation


print("Gradio XAI wrapper created successfully.")

In [ ]:
# ============================================================
# GRADIO AGRICULTURE AI + XAI DASHBOARD
# ============================================================

import gradio as gr

with gr.Blocks(title="Agriculture AI - 7 Module XAI System") as xai_demo:

    gr.Markdown(
        """
        # 🌾 Agriculture AI — Integrated 7-Module XAI System

        Enter agricultural and environmental parameters below.
        The system will generate predictions from the existing
        7 AI models and provide input-specific explanations.
        """
    )

    # --------------------------------------------------------
    # INPUTS
    # --------------------------------------------------------

    with gr.Row():

        with gr.Column():

            gr.Markdown("### 🌱 Crop & Environment")

            N = gr.Number(label="Nitrogen (N)", value=90)
            P = gr.Number(label="Phosphorus (P)", value=42)
            K = gr.Number(label="Potassium (K)", value=43)

            temperature = gr.Number(
                label="Temperature (°C)",
                value=25
            )

            humidity = gr.Number(
                label="Humidity (%)",
                value=80
            )

            ph = gr.Number(
                label="Soil pH",
                value=6.5
            )

            rainfall = gr.Number(
                label="Rainfall (mm)",
                value=200
            )

        with gr.Column():

            gr.Markdown("### 💧 Soil & Farm")

            soil_moisture = gr.Number(
                label="Soil Moisture",
                value=200
            )

            soil_temperature = gr.Number(
                label="Soil Temperature (°C)",
                value=25
            )

            crop_year = gr.Number(
                label="Crop Year",
                value=2025
            )

            area = gr.Number(
                label="Area",
                value=100
            )

            fertilizer = gr.Number(
                label="Fertilizer",
                value=50000
            )

            pesticide = gr.Number(
                label="Pesticide",
                value=500
            )

        with gr.Column():

            gr.Markdown("### 🌦️ Location & Weather")

            season = gr.Dropdown(
                choices=[
                    "Kharif",
                    "Rabi",
                    "Whole Year",
                    "Autumn",
                    "Summer",
                    "Winter",
                    "Monsoon",
                    "Post-monsoon"
                ],
                value="Kharif",
                label="Season"
            )

            state = gr.Dropdown(
                choices=[
                    "Andhra Pradesh",
                    "Telangana",
                    "Tamil Nadu",
                    "Karnataka",
                    "Maharashtra",
                    "Gujarat",
                    "Punjab",
                    "Haryana",
                    "Uttar Pradesh",
                    "West Bengal",
                    "Odisha",
                    "Bihar",
                    "Rajasthan",
                    "Kerala"
                ],
                value="Andhra Pradesh",
                label="State"
            )

            month = gr.Dropdown(
                choices=[
                    "January",
                    "February",
                    "March",
                    "April",
                    "May",
                    "June",
                    "July",
                    "August",
                    "September",
                    "October",
                    "November",
                    "December"
                ],
                value="July",
                label="Month"
            )

            wind_speed = gr.Number(
                label="Wind Speed",
                value=10
            )

            air_pressure = gr.Number(
                label="Air Pressure",
                value=1010
            )

    # --------------------------------------------------------
    # BUTTON
    # --------------------------------------------------------

    predict_button = gr.Button(
        "🔍 Predict & Explain",
        variant="primary"
    )

    # --------------------------------------------------------
    # OUTPUTS
    # --------------------------------------------------------

    gr.Markdown("## 📊 Predictions")

    summary_output = gr.Textbox(
        label="Agriculture AI Results",
        lines=15
    )

    explanation_output = gr.Textbox(
        label="🔎 Explainable AI — Why did the model make this prediction?",
        lines=35
    )

    # --------------------------------------------------------
    # CONNECT BUTTON
    # --------------------------------------------------------

    predict_button.click(
        fn=gradio_xai_prediction,

        inputs=[
            N, P, K,
            temperature, humidity, ph, rainfall,
            soil_moisture, soil_temperature,
            crop_year, area, fertilizer, pesticide,
            season, state, month,
            wind_speed, air_pressure
        ],

        outputs=[
            summary_output,
            explanation_output
        ]
    )


print("Gradio XAI dashboard created successfully.")

In [ ]:
# ============================================================
# LAUNCH FINAL AGRICULTURE AI + XAI DASHBOARD
# ============================================================

xai_demo.launch(
    share=True,
    debug=True
)